# Volatility Environment Phase 2
固定28戦略・16,298 tradesの5分位診断。閾値最適化・Entry除外・Risk配分・live変更なし。
計画SHA: 1acc1fa530610eafcd985d446ba86c4dc415b5d6（実装前remote確認済み）。
Q1=[0,20), Q2=[20,40), Q3=[40,60), Q4=[60,80), Q5=[80,100]。
Primary ATR20 SMA、Robustness RV20 ddof=1、JST日次、過去252日のmidrankはPhase 1の不変コードを再利用。
正式支持条件: pooled Q5>Q1、Spearman>0、隣接増加3/4以上、等重みQ5>Q1かつSpearman>0。CIは補助。
個別正式判定・等重みの対象は全5cell各20件以上。2022–2026は既閲覧。


## 実行結果
この実装コミット時点では実データ未実行です。結果固定時にQ1→Q5の主要表を本文へ追記します。
再実行結果は下の表示セルに表示されます。RESEARCH_SHAにremote確認済み実装SHAを設定してください。


In [ ]:
from pathlib import Path
RESEARCH_SHA = ""  # 結果版Notebookでは検証済み実装SHAを固定
PHASE1_SHA = "cafbb6ff0bfe81e439b80ed62d50498494814e86"
BASELINE = Path("/content/drive/MyDrive/time-entry-portfolio-lab/daily_stop/baseline_cc32f32e3df5/daily_stop_baseline_trades.csv")
M1_ROOT = Path("/content/drive/MyDrive")
OUT = Path("/content")
MOUNT_INPUT_DRIVE = True
if MOUNT_INPUT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
import urllib.request, sys, subprocess, hashlib
assert len(RESEARCH_SHA)==40 and all(c in '0123456789abcdef' for c in RESEARCH_SHA)
stage = Path('/content/volatility_phase2_code'); stage.mkdir(exist_ok=True)
reference = stage/'phase1_reference'; reference.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/TR-KJ/time-entry-portfolio-lab/'
for name in ['src/research/volatility_phase1.py','src/research/volatility_phase1_frozen_inputs.json','tests/test_volatility_phase1.py','tests/verify_volatility_phase1.py']:
    (stage/Path(name).name).write_bytes(urllib.request.urlopen(base+PHASE1_SHA+'/'+name).read())
for name in ['src/research/volatility_phase2.py','tests/test_volatility_phase2.py','tests/verify_volatility_phase2.py']:
    (stage/Path(name).name).write_bytes(urllib.request.urlopen(base+RESEARCH_SHA+'/'+name).read())
for name in ['strategy_primary','strategy_robustness','group_summary','period_summary','decision','combined_decision','regime_coverage','assignment_audit_light','input_manifest']:
    filename=f'volatility_phase1_{name}.csv'
    (reference/filename).write_bytes(urllib.request.urlopen(base+PHASE1_SHA+'/results/volatility_phase1/'+filename).read())
sys.path.insert(0,str(stage))
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(stage),'-p','test_volatility_phase*.py','-v'],check=True)
from volatility_phase2 import run
result = run(BASELINE,M1_ROOT,OUT,RESEARCH_SHA,reference)


In [ ]:
from IPython.display import display
print('Portfolio Q1 → Q5: trade-weighted / strategy-equal-weighted')
display(result['portfolio_quintiles'])
summary=result['monotonicity_summary']
display(summary[(summary.Period=='FULL') & (summary.Scope=='Portfolio')])
print('全28戦略')
display(result['strategy_quintiles'])
print('事前指定4戦略（1/6/12/23）')
display(summary[(summary.Period=='FULL') & summary.PrespecifiedSubgroup])
print('補助群')
display(result['group_quintiles'])
display(summary[(summary.Period=='FULL') & (summary.ScopeType=='Group')])
for name in ['coverage','phase1_regression','verification','manual_audit','run_record']:
    print(name); display(result[name])


## 解釈と次段階
CIは暦週cluster bootstrap 5000回 seed=20260913。週内の依存を保持するが週を跨ぐ依存・非定常性は完全には扱わない。多重比較未調整。
両方式のPortfolioが支持された場合のみPhase 3を別事前登録として検討する。LOWでもAvgR正のPhase 1結果からRisk Allocationを優先仮説に、Entry Filterとの比較を検討する段階まで。
今回の結果で運用採用・閾値最適化・Money Simulationは行わない。完全版assignments/dailyはGitHubへアップロードしない。


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil
    target=Path('/content/drive/MyDrive/time-entry-portfolio-lab/volatility_phase2')
    target.mkdir(parents=True,exist_ok=True)
    for p in OUT.glob('volatility_phase2_*.csv'):
        shutil.copy2(p,target/p.name)
